# Hidden-State Agreement Across Language Models
### Experiment 01 · Reproducible cross-layer analysis

**Research question.** Do independently trained language models organize the same sentences into similar nearest-neighbor structures?

**Models:** Pythia-160M (decoder), BERT-base (encoder), MiniLM-L6 (sentence-embedding model), plus randomly initialized controls. **Dataset:** 2,000 English AG News examples; **primary measure:** top-10 neighborhood overlap.

This notebook is a cleaned, runnable reconstruction of the **seven experiments** in the original `hidden_state_agreement_colab(2).ipynb`. Earlier exploration (`Untitled5.ipynb`) informed the study; the multi-model extension (`Untitled7(1).ipynb`) will belong to a separate notebook. Results are generated by running the code below; no historical outputs are presented as new results.

**How to run:** open in Google Colab → Runtime → Run all. A GPU is preferable, though CPU can work. Download the `outputs/` files using the final cell. The first run downloads pretrained model weights and the dataset.

**Interpretation warning:** geometric agreement is not proof of shared semantic features or reasoning processes.

## 1. Environment and reproducibility
Dependencies and parameters are defined once. A small `QUICK` toggle exists only for debugging; the **default is the full 2,000-text experiment**.

In [ ]:
%pip install -q "transformers>=4.45,<5" datasets scikit-learn matplotlib pandas

In [ ]:
import gc
import json
import os
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from transformers import AutoConfig, AutoModel, AutoTokenizer

QUICK = False  # Set True only for a quick debugging run

SEED = 0
N_TEXTS = 160 if QUICK else 2000
N_RANDOM_SEEDS = 1 if QUICK else 2
K = 10
K_CLASS = 5
BATCH = 16  # Conservatively preserve the original extraction behavior
GPU_METRICS = True  # Use CUDA for kNN and overlap when available; CPU fallback otherwise
MAX_LEN = 128
OUT = "outputs_quick" if QUICK else "outputs"

MODELS = {
    "pythia": "EleutherAI/pythia-160m",
    "bert": "bert-base-uncased",
    "minilm": "sentence-transformers/all-MiniLM-L6-v2",
}
DISPLAY = {"pythia": "Pythia-160M", "bert": "BERT-base", "minilm": "MiniLM-L6"}
PAIR = ("pythia", "bert")

os.makedirs(OUT, exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("device:", device, "| texts:", N_TEXTS, "| random seeds:", N_RANDOM_SEEDS)
print("GPU metrics:", GPU_METRICS and device == "cuda", "| extraction precision: float32")

REPORT = []
RESULTS = {}


def log(text=""):
    print(text)
    REPORT.append(text)


def save_fig(name):
    plt.savefig(f"{OUT}/{name}.png", dpi=150, bbox_inches="tight")
    plt.show()

## 2. Dataset
We shuffle with a fixed seed and use a fixed 50/50 *exploration/confirmation* split. The exploration half is used to select a layer pair; its apparent best score must **not** be reported as the held-out result.

In [ ]:
ds = load_dataset("fancyzhx/ag_news", split="train").shuffle(seed=SEED).select(range(N_TEXTS))
texts = list(ds["text"])
labels = np.array(ds["label"])
half = N_TEXTS // 2
EXPLORE = np.arange(0, half)
CONFIRM = np.arange(half, N_TEXTS)
print("class counts:", np.bincount(labels))
print("example:", texts[0][:140])

## 3. Extract per-layer sentence representations
Mean-pool hidden states over nonpadding tokens for each transformer layer (including the embedding layer). All models are frozen and in evaluation mode. Random controls use the same architecture with randomly initialized weights.

The full extraction is intentionally preserved from the original validated experiment, except for small robustness checks.

**Precision policy:** Model inference still uses the original FP32 precision, batch size 16, and mean pooling; no mixed precision or altered inference parameters are introduced.


In [ ]:
def extract_layers(name, random_weights=False, seed=0):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token or tok.unk_token
        if tok.pad_token is None:
            raise ValueError(f"Tokenizer {name} has no usable padding token")
    if random_weights:
        torch.manual_seed(seed)
        model = AutoModel.from_config(AutoConfig.from_pretrained(name))
    else:
        model = AutoModel.from_pretrained(name)
    model = model.to(device).eval()

    per_layer = None
    for s in range(0, len(texts), BATCH):
        inputs = tok(texts[s:s + BATCH], padding=True, truncation=True,
                     max_length=MAX_LEN, return_tensors="pt").to(device)
        with torch.inference_mode():
            hs = model(**inputs, output_hidden_states=True).hidden_states
            m = inputs["attention_mask"].unsqueeze(-1).to(hs[0].dtype)
            reps = [((h * m).sum(1) / m.sum(1)).float().cpu() for h in hs]
        if per_layer is None:
            per_layer = [[] for _ in reps]
        for i, r in enumerate(reps):
            per_layer[i].append(r)
    del model, tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return [torch.cat(x) for x in per_layer]


LAYERS = {}
t0 = time.time()
for key, name in MODELS.items():
    print("extracting", key, end=" ... ")
    LAYERS[key] = extract_layers(name)
    print(len(LAYERS[key]), "layers")
    for s in range(N_RANDOM_SEEDS):
        LAYERS[f"rand_{key}_{s}"] = extract_layers(name, random_weights=True, seed=100 + s)
print(f"done in {time.time() - t0:.0f}s")

## 4. Metrics and plotting helpers
**kNN overlap** is the fraction of shared top-*k* nearest neighbors for each sentence. Neighborhoods use cosine similarity after dimension-wise z-scoring. **CKA** provides a complementary measure of representation similarity. **TF-IDF** serves as a lexical baseline.

For two independent random neighbor sets of size *k* among *N−1* candidates, the expected overlap fraction is `k/(N-1)`; this is a reference point, not a formal statistical significance test.

**Normalization:** the z-score is computed separately within each evaluated subset. It prevents a few large-magnitude coordinates from dominating cosine similarity, but changes the geometry being measured.

**Execution optimization (no change in research design):** Exact cosine top-k and neighbor-overlap calculations can run on CUDA in FP32. CKA uses the same centered linear CKA formula and FP32, but processes layer pairs separately to cap peak GPU memory. `GPU_METRICS=False` provides a CPU fallback. Fixed data selection, seeds, pooling, normalization, metrics, and experiments remain unchanged. CUDA vs CPU may produce tiny floating-point differences and different tie ordering for equal similarities.


In [ ]:
def standardize(X):
    X = X.float()
    return (X - X.mean(0)) / (X.std(0) + 1e-6)


def knn_idx(X, k):
    """Exact cosine kNN (same formula as the original); CUDA if available.

    Return indices on CPU to keep notebook memory usage small and preserve
    compatibility with existing pandas/NumPy reporting code.
    """
    calc_device = device if GPU_METRICS else "cpu"
    X = F.normalize(X.to(calc_device, dtype=torch.float32), dim=1)
    sim = X @ X.T
    sim.fill_diagonal_(-float("inf"))
    result = sim.topk(k, dim=1).indices.cpu()
    del sim, X
    return result


def per_sentence_overlap(nx, ny):
    """Exact shared-neighbor fraction per sentence, preserving old semantics."""
    if nx.device != ny.device:
        ny = ny.to(nx.device)
    k = nx.shape[1]
    return (nx.unsqueeze(2) == ny.unsqueeze(1)).any(dim=2).float().sum(1) / k


def bank(layers, idx=None, k=K, std=True):
    out = []
    for X in layers:
        if idx is not None:
            X = X[idx]
        out.append(knn_idx(standardize(X) if std else X, k))
    return out


def agreement_matrix(na, nb):
    """Pairwise kNN overlap, evaluated on GPU without changing the metric."""
    calc_device = device if GPU_METRICS else "cpu"
    A = [x.to(calc_device) for x in na]
    B = [x.to(calc_device) for x in nb]
    M = np.empty((len(A), len(B)), dtype=np.float64)
    for i, x in enumerate(A):
        for j, y in enumerate(B):
            M[i, j] = per_sentence_overlap(x, y).mean().item()
    return M


_tfidf = TfidfVectorizer(stop_words="english").fit(texts)


def tfidf_idx(idx=None, k=K):
    docs = texts if idx is None else [texts[i] for i in idx]
    Xs = _tfidf.transform(docs)
    sim = torch.tensor((Xs @ Xs.T).toarray(), dtype=torch.float32)
    sim.fill_diagonal_(-float("inf"))
    return sim.topk(k, dim=1).indices


def cka_matrix(la, lb, std=False):
    """Linear CKA in FP32, computing one layer pair at a time to bound VRAM.

    This uses the same feature centering and Frobenius-norm definition as
    the original notebook (and does not switch to half precision).
    """
    def prepare(X):
        X = standardize(X) if std else X.float()
        X = X - X.mean(0, keepdim=True)
        return X.to(device)

    M = np.empty((len(la), len(lb)), dtype=np.float64)
    # Store the B layers on CPU; move one onto GPU per layer pair.
    B = [prepare(x).cpu() for x in lb]
    norm_b = [(torch.linalg.matrix_norm(y.T @ y).item()) for y in B]
    for i, x in enumerate(la):
        A = prepare(x)
        norm_a = torch.linalg.matrix_norm(A.T @ A)
        for j, y_cpu in enumerate(B):
            Y = y_cpu.to(device)
            xy = torch.linalg.matrix_norm(A.T @ Y) ** 2
            M[i, j] = (xy / (norm_a * norm_b[j] + 1e-12)).item()
            del Y
        del A
    return M


def boot_ci(v, n=2000, seed=0):
    v = np.asarray(v, dtype=float)
    rng = np.random.default_rng(seed)
    means = v[rng.integers(0, len(v), (n, len(v)))].mean(1)
    return float(v.mean()), float(np.percentile(means, 2.5)), float(np.percentile(means, 97.5))


def layer_at(n_layers, depth):
    return int(round(depth * (n_layers - 1)))


def depth_grid(n):
    return np.linspace(0, 1, n)


def depth_bins(M, edges=(0, 0.1, 0.25, 0.5, 1.01)):
    da, db = depth_grid(M.shape[0]), depth_grid(M.shape[1])
    dd = np.abs(da[:, None] - db[None, :])
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (dd >= lo) & (dd < hi)
        rows.append((f"{lo:.2f}-{min(hi, 1.0):.2f}", float(M[mask].mean()), int(mask.sum())))
    return rows


def random_mean(key_b):
    return [LAYERS[f"rand_{key_b}_{s}"] for s in range(N_RANDOM_SEEDS)]


def heatmap(ax, M, title, xlabel, ylabel, vmin=None, vmax=None, cmap="viridis"):
    im = ax.imshow(M, origin="lower", aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_xticks(range(M.shape[1]))
    ax.set_yticks(range(M.shape[0]))
    plt.colorbar(im, ax=ax, fraction=0.046)

## 5. Precompute nearest-neighbor sets
Reuse these indices for comparisons to avoid redundant similarity calculations.

In [ ]:
NBR = {k: bank(v) for k, v in LAYERS.items()}
NBR_TFIDF = tfidf_idx()
chance = K / (N_TEXTS - 1)
log(f"SETUP: {N_TEXTS} AG News texts, k={K}, chance ~ {chance:.1%}, "
    f"models: {', '.join(DISPLAY[k] for k in MODELS)}, random seeds: {N_RANDOM_SEEDS}")
log("Representation: per-layer mean pooling, z-scored dimensions unless stated otherwise.")
log()

## Experiment 1 — Cross-layer agreement
Compare all Pythia × BERT layer pairs against random-initialized BERT controls. The heatmaps show where agreement is stronger; their highest cells are **exploratory**, not independent evidence.

In [ ]:
a, b = PAIR
M_tr = agreement_matrix(NBR[a], NBR[b])
M_rd = np.mean([agreement_matrix(NBR[a], NBR[f"rand_{b}_{s}"]) for s in range(N_RANDOM_SEEDS)], axis=0)
vmax = max(M_tr.max(), M_rd.max())

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
heatmap(axes[0], M_tr, f"{DISPLAY[a]} vs {DISPLAY[b]} (trained)", f"{DISPLAY[b]} layer", f"{DISPLAY[a]} layer", 0, vmax)
heatmap(axes[1], M_rd, f"{DISPLAY[a]} vs random {DISPLAY[b]}", f"random {DISPLAY[b]} layer", f"{DISPLAY[a]} layer", 0, vmax)
heatmap(axes[2], M_tr - M_rd, "trained minus random", f"{DISPLAY[b]} layer", f"{DISPLAY[a]} layer", cmap="magma")
plt.suptitle(f"Experiment 1: kNN overlap (k={K}, chance {chance:.1%})")
plt.tight_layout()
save_fig("exp1_heatmaps")

diff = M_tr - M_rd
log("EXPERIMENT 1: cross-layer agreement, " + f"{DISPLAY[a]} x {DISPLAY[b]}")
log(f"  trained: mean {M_tr.mean():.3f}  min {M_tr.min():.3f}  max {M_tr.max():.3f}")
log(f"  random control: mean {M_rd.mean():.3f}  max {M_rd.max():.3f}")
log(f"  trained minus random: mean {diff.mean():+.3f}, trained higher in {(diff > 0).mean():.0%} of layer pairs")
top = np.argsort(M_tr, axis=None)[::-1][:5]
log("  five highest layer pairs (descriptive only; see Experiment 6 for held-out confirmation):")
for t in top:
    i, j = np.unravel_index(t, M_tr.shape)
    log(f"    {DISPLAY[a]} {i:2d} x {DISPLAY[b]} {j:2d}: {M_tr[i, j]:.3f}")
log()
RESULTS["exp1"] = {"trained_mean": float(M_tr.mean()), "random_mean": float(M_rd.mean()),
                   "trained_gt_random_fraction": float((diff > 0).mean()),
                   "trained_max": float(M_tr.max())}

**Reading the result:** Interpret the displayed graph and printed values, but avoid treating correlated layer-pair cells or a single dataset as independent replications.

## Experiment 2 — Relative model depth
Group layer pairs by their relative-depth difference. This tests whether agreement clusters around similarly positioned layers.

In [ ]:
rows = []
for (a2, b2) in [("pythia", "bert"), ("pythia", "minilm"), ("bert", "minilm")]:
    Mt = agreement_matrix(NBR[a2], NBR[b2])
    Mr = np.mean([agreement_matrix(NBR[a2], NBR[f"rand_{b2}_{s}"]) for s in range(N_RANDOM_SEEDS)], axis=0)
    for (label, vt, n), (_, vr, _) in zip(depth_bins(Mt), depth_bins(Mr)):
        rows.append({"pair": f"{DISPLAY[a2]} x {DISPLAY[b2]}", "|depth difference|": label,
                     "trained": vt, "random": vr, "cells": n})
depth_df = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
for ax, (pair, g) in zip(axes, depth_df.groupby("pair", sort=False)):
    ax.plot(g["|depth difference|"], g["trained"], marker="o", label="trained")
    ax.plot(g["|depth difference|"], g["random"], marker="o", label="random control")
    ax.axhline(chance, color="gray", ls=":", label="chance")
    ax.set_title(pair, fontsize=10)
    ax.set_xlabel("|relative depth difference|")
axes[0].set_ylabel(f"mean kNN overlap (k={K})")
axes[0].legend()
plt.suptitle("Experiment 2: agreement vs relative depth gap")
plt.tight_layout()
save_fig("exp2_depth")

log("EXPERIMENT 2: mean agreement by relative-depth gap (trained vs random control)")
log(depth_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
log()
RESULTS["exp2"] = depth_df.to_dict(orient="records")

**Reading the result:** Interpret the displayed graph and printed values, but avoid treating correlated layer-pair cells or a single dataset as independent replications.

## Experiment 3 — Lexical baseline
Compare hidden-state neighborhoods with TF-IDF neighborhoods to assess how much of the apparent agreement may be explained by word overlap.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
tf_rows = []
for key in MODELS:
    vals = [per_sentence_overlap(n, NBR_TFIDF).mean().item() for n in NBR[key]]
    ax.plot(depth_grid(len(vals)), vals, marker="o", label=f"{DISPLAY[key]} vs TF-IDF")
    tf_rows.append({"model": DISPLAY[key], "min": min(vals), "max": max(vals), "mean": float(np.mean(vals))})
ax.axhline(M_tr.mean(), color="black", ls="--", label=f"{DISPLAY[a]}-{DISPLAY[b]} mean")
ax.axhline(chance, color="gray", ls=":", label="chance")
ax.set_xlabel("relative depth")
ax.set_ylabel(f"kNN overlap (k={K})")
ax.set_title("Experiment 3: agreement with a TF-IDF baseline")
ax.legend(fontsize=8)
save_fig("exp3_tfidf")

best_pair = np.unravel_index(np.argmax(M_tr), M_tr.shape)
lex = max(per_sentence_overlap(NBR[a][best_pair[0]], NBR_TFIDF).mean().item(),
          per_sentence_overlap(NBR[b][best_pair[1]], NBR_TFIDF).mean().item())
log("EXPERIMENT 3: agreement with TF-IDF neighbours")
log(pd.DataFrame(tf_rows).to_string(index=False, float_format=lambda x: f"{x:.3f}"))
log(f"  highest {DISPLAY[a]}-{DISPLAY[b]} pair: {M_tr[best_pair]:.3f} vs "
    f"{lex:.3f} for the more lexical of its two layers ({M_tr[best_pair] / lex:.2f}x)")
log()
RESULTS["exp3"] = {"tfidf": tf_rows, "best_pair_vs_lexical_ratio": float(M_tr[best_pair] / lex)}

**Reading the result:** Interpret the displayed graph and printed values, but avoid treating correlated layer-pair cells or a single dataset as independent replications.

## Experiment 4 — Sensitivity to metric and scaling
Compare raw and standardized representations under kNN overlap and linear CKA. These metrics are not interchangeable.

In [ ]:
variants = {}
for std in (False, True):
    tag = "standardized" if std else "raw"
    nb_t = NBR[b] if std else bank(LAYERS[b], std=False)
    nb_a = NBR[a] if std else bank(LAYERS[a], std=False)
    knn_t = agreement_matrix(nb_a, nb_t)
    knn_r = np.mean([agreement_matrix(nb_a, bank(LAYERS[f"rand_{b}_{s}"], std=std))
                     for s in range(N_RANDOM_SEEDS)], axis=0)
    cka_t = cka_matrix(LAYERS[a], LAYERS[b], std=std)
    cka_r = np.mean([cka_matrix(LAYERS[a], LAYERS[f"rand_{b}_{s}"], std=std)
                     for s in range(N_RANDOM_SEEDS)], axis=0)
    variants[f"kNN, {tag}"] = (knn_t, knn_r)
    variants[f"CKA, {tag}"] = (cka_t, cka_r)

check_rows = []
for name, (t, r) in variants.items():
    check_rows.append({"metric": name, "trained mean": t.mean(), "random mean": r.mean(),
                       "mean diff": (t - r).mean(), "trained > random": f"{(t > r).mean():.0%}"})
check_df = pd.DataFrame(check_rows)

fig, axes = plt.subplots(1, 4, figsize=(19, 4.2))
for ax, (name, (t, r)) in zip(axes, variants.items()):
    heatmap(ax, t - r, f"{name}: trained minus random", f"{DISPLAY[b]} layer", f"{DISPLAY[a]} layer",
            cmap="coolwarm", vmin=-np.abs(t - r).max(), vmax=np.abs(t - r).max())
plt.suptitle("Experiment 4: metric sanity check (red = trained above random, blue = random above trained)")
plt.tight_layout()
save_fig("exp4_metric_check")

log("EXPERIMENT 4: does the metric rank trained models above a random network?")
log(check_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
log()
RESULTS["exp4"] = check_df.to_dict(orient="records")

**Reading the result:** Interpret the displayed graph and printed values, but avoid treating correlated layer-pair cells or a single dataset as independent replications.

## Experiment 5 — Within-topic comparison
Recompute neighbors *within* each AG News label category to see whether agreement persists after conditioning on broad topic. Topic is controlled coarsely, not eliminated as a possible explanation.

In [ ]:
rows = []
for depth in (0.25, 0.5, 0.75, 1.0):
    ia, ib = layer_at(len(LAYERS[a]), depth), layer_at(len(LAYERS[b]), depth)
    for c in np.unique(labels):
        idx = np.where(labels == c)[0]
        na_ = knn_idx(standardize(LAYERS[a][ia][idx]), K_CLASS)
        nb_ = knn_idx(standardize(LAYERS[b][ib][idx]), K_CLASS)
        nr_ = [knn_idx(standardize(LAYERS[f"rand_{b}_{s}"][ib][idx]), K_CLASS) for s in range(N_RANDOM_SEEDS)]
        nt_ = tfidf_idx(idx, K_CLASS)
        rows.append({"rel. depth": depth, "topic": int(c), "n": len(idx),
                     "trained": per_sentence_overlap(na_, nb_).mean().item(),
                     "random BERT": float(np.mean([per_sentence_overlap(na_, r).mean().item() for r in nr_])),
                     f"{DISPLAY[a]}-TFIDF": per_sentence_overlap(na_, nt_).mean().item(),
                     f"{DISPLAY[b]}-TFIDF": per_sentence_overlap(nb_, nt_).mean().item()})
topic_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 4))
g = topic_df.groupby("rel. depth")[["trained", "random BERT", f"{DISPLAY[a]}-TFIDF", f"{DISPLAY[b]}-TFIDF"]].mean()
g.plot(marker="o", ax=ax)
ax.axhline(K_CLASS / (len(texts) / 4), color="gray", ls=":")
ax.set_ylabel(f"within-topic kNN overlap (k={K_CLASS})")
ax.set_title("Experiment 5: agreement inside a single topic")
save_fig("exp5_within_topic")

log(f"EXPERIMENT 5: within-topic agreement (k={K_CLASS}), averaged over topics")
log(g.to_string(float_format=lambda x: f"{x:.3f}"))
log()
RESULTS["exp5"] = g.reset_index().to_dict(orient="records")

**Reading the result:** Interpret the displayed graph and printed values, but avoid treating correlated layer-pair cells or a single dataset as independent replications.

## Experiment 6 — Held-out selection and uncertainty
Select the best layer pair using only the exploration split. Evaluate it on the untouched confirmation split and compute bootstrap confidence intervals for mean per-sentence overlap. These intervals reflect resampling of this dataset, not variability across retraining seeds.

In [ ]:
nb_a_exp = bank(LAYERS[a], EXPLORE)
nb_b_exp = bank(LAYERS[b], EXPLORE)
M_exp = agreement_matrix(nb_a_exp, nb_b_exp)
ia, ib = np.unravel_index(np.argmax(M_exp), M_exp.shape)

na_c = knn_idx(standardize(LAYERS[a][ia][CONFIRM]), K)
nb_c = knn_idx(standardize(LAYERS[b][ib][CONFIRM]), K)
nt_c = tfidf_idx(CONFIRM, K)
nr_c = [knn_idx(standardize(LAYERS[f"rand_{b}_{s}"][ib][CONFIRM]), K) for s in range(N_RANDOM_SEEDS)]

series = {
    f"{DISPLAY[a]} x {DISPLAY[b]} (selected pair)": per_sentence_overlap(na_c, nb_c).numpy(),
    f"{DISPLAY[a]} x random {DISPLAY[b]}": np.mean([per_sentence_overlap(na_c, r).numpy() for r in nr_c], axis=0),
    f"{DISPLAY[a]} x TF-IDF": per_sentence_overlap(na_c, nt_c).numpy(),
    f"{DISPLAY[b]} x TF-IDF": per_sentence_overlap(nb_c, nt_c).numpy(),
}
conf_chance = K / (len(CONFIRM) - 1)
ci_rows = []
for name, v in series.items():
    m, lo, hi = boot_ci(v)
    ci_rows.append({"comparison": name, "mean": m, "ci_low": lo, "ci_high": hi})
ci_df = pd.DataFrame(ci_rows)

fig, ax = plt.subplots(figsize=(8, 3.6))
ypos = np.arange(len(ci_df))[::-1]
ax.errorbar(ci_df["mean"], ypos, xerr=[ci_df["mean"] - ci_df["ci_low"], ci_df["ci_high"] - ci_df["mean"]],
            fmt="o", capsize=4)
ax.set_yticks(ypos)
ax.set_yticklabels(ci_df["comparison"])
ax.axvline(conf_chance, color="gray", ls=":", label="chance")
ax.set_xlabel(f"kNN overlap on held-out half (k={K}), 95% bootstrap CI")
ax.set_title(f"Experiment 6: pair chosen on explore half: {DISPLAY[a]} layer {ia}, {DISPLAY[b]} layer {ib}")
ax.legend()
save_fig("exp6_heldout")

log("EXPERIMENT 6: held-out confirmation")
log(f"  pair selected on explore half: {DISPLAY[a]} layer {ia} x {DISPLAY[b]} layer {ib} "
    f"(explore score {M_exp[ia, ib]:.3f})")
log(ci_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
log(f"  chance on held-out half ~ {conf_chance:.1%}")
log()
RESULTS["exp6"] = {"selected": [int(ia), int(ib)], "explore_score": float(M_exp[ia, ib]),
                   "confirm": ci_df.to_dict(orient="records")}

**Reading the result:** Interpret the displayed graph and printed values, but avoid treating correlated layer-pair cells or a single dataset as independent replications.

## Experiment 7 — Third-model comparison
Compare matched-depth agreement for Pythia, BERT and MiniLM, each versus a random-initialized control. The maximum cell is shown for context but not used as the primary matched-depth estimate.

In [ ]:
rows = []
for (x, y) in [("pythia", "bert"), ("pythia", "minilm"), ("bert", "minilm")]:
    Mt = agreement_matrix(NBR[x], NBR[y])
    Mr = np.mean([agreement_matrix(NBR[x], NBR[f"rand_{y}_{s}"]) for s in range(N_RANDOM_SEEDS)], axis=0)
    dx, dy = depth_grid(Mt.shape[0]), depth_grid(Mt.shape[1])
    mask = np.abs(dx[:, None] - dy[None, :]) < 0.15
    rows.append({"pair": f"{DISPLAY[x]} x {DISPLAY[y]}",
                 "matched-depth trained": Mt[mask].mean(),
                 "matched-depth random": Mr[mask].mean(),
                 "ratio": Mt[mask].mean() / Mr[mask].mean(),
                 "highest cell": Mt.max()})
pair_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 3.6))
w = 0.38
xs = np.arange(len(pair_df))
ax.bar(xs - w / 2, pair_df["matched-depth trained"], w, label="trained")
ax.bar(xs + w / 2, pair_df["matched-depth random"], w, label="random control")
ax.axhline(chance, color="gray", ls=":", label="chance")
ax.set_xticks(xs)
ax.set_xticklabels(pair_df["pair"], fontsize=8)
ax.set_ylabel(f"mean kNN overlap (k={K})")
ax.set_title("Experiment 7: matched-depth agreement across model pairs")
ax.legend()
save_fig("exp7_three_models")

log("EXPERIMENT 7: matched-depth agreement for each model pair")
log(pair_df.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
log()
RESULTS["exp7"] = pair_df.to_dict(orient="records")

**Reading the result:** Interpret the displayed graph and printed values, but avoid treating correlated layer-pair cells or a single dataset as independent replications.

## 6. Save results and review limitations
The generated figures, JSON metrics and text report are written to `outputs/`. Only results computed by this run enter those files.

In [ ]:
log("LIMITATIONS")
log("  - One dataset (AG News, English, 4 topics), mean pooling only, k-NN overlap as the main metric.")
log("  - Agreement in neighbour structure does not by itself show shared meaning or shared features.")
log(f"  - {N_RANDOM_SEEDS} random-initialisation seed(s) per architecture; one data sample.")
log("  - Hidden states were standardized per dimension; raw results are reported in Experiment 4.")

with open(f"{OUT}/report.txt", "w") as f:
    f.write("\n".join(REPORT))
with open(f"{OUT}/results.json", "w") as f:
    json.dump(RESULTS, f, indent=2, default=float)
print(f"\nSaved report.txt, results.json and figures to ./{OUT}/")

### Research limitations and next steps

- The dataset is English news in four categories; it does not establish generalization to other domains or languages.
- Standardized mean-pooling may favor some architectures or layers; padding is masked, but different tokenizers and training objectives remain confounds.
- Random initialization controls for architecture without learning, not for all possible baselines; two random seeds are limited.
- A high kNN overlap does **not** prove two models encode identical concepts.
- Choosing a layer pair on held-out data is avoided in Experiment 6, but repeated notebook development can still influence researcher decisions.
- Future work: multiple datasets/seeds, stronger lexical/topic controls, representation alignment and task-level evaluation.

The companion notebook will extend this study to more architectures and downstream STS-B/PAWS analyses, drawing from the original `Untitled7(1).ipynb`.

## 7. Download the generated report (Google Colab)

In [ ]:
# Packages are installed by the first cell. This works in Colab; elsewhere the ZIP is saved locally.
import shutil
archive_path = shutil.make_archive(OUT, "zip", OUT)
print("Saved:", archive_path)
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print("Download manually from your notebook environment.")